In [10]:
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 

In [6]:
%pip install pandas

import re
import pandas as pd

with open("aicdapub.html", "r", encoding="utf-8") as f:
    text = f.read()

# --- Substitutions ---
subs = re.findall(r'c\.([0-9]+)([ACGT])>([ACGT])', text)

# --- Deletions ---
dels = re.findall(r'c\.([0-9]+)(?:_([0-9]+))?del([ACGT]+)?', text)

# Prepare data for DataFrame
mutation_data = []

print("Substitutions:")
for pos, ref, alt in set(subs):
    print(f"Position {pos}, Reference base {ref}")
    mutation_data.append({
        'mutation type': 'Substitution',
        'position': pos,
        'reference base/sequence': ref
    })

print("\nDeletions:")
for start, end, seq in set(dels):
    if end:
        position_str = f"{start}-{end}"
        print(f"Positions {position_str}, Deleted sequence: {seq}")
    else:
        position_str = start
        print(f"Position {position_str}, Deleted base: {seq}")
    mutation_data.append({
        'mutation type': 'Deletion',
        'position': position_str,
        'reference base/sequence': seq
    })

# Create DataFrame
df_mutations = pd.DataFrame(mutation_data)

# Save to CSV
df_mutations.to_csv('AICDA_mut.csv', index=False)

print("\nMutation data saved to AICDA_mut.csv")
display(df_mutations.head())

Note: you may need to restart the kernel to use updated packages.


FileNotFoundError: [Errno 2] No such file or directory: 'aicdapub.html'

In [ ]:
%pip install openpyxl

In [ ]:
df_mutations.info()

# Task
`!pip install biopython`

## Install Biopython

### Subtask:
Install the Biopython library, which is essential for parsing FASTA files easily.


**Reasoning**:
To install the Biopython library as instructed, I will use a code block to run the `!pip install biopython` command.



In [ ]:
%pip install biopython

# Task
Implement a function to iterate through each FASTA sequence and cross-reference it with the `df_mutations` data. For each mutation, the function will verify if the base or sequence at the given position(s) in the FASTA file matches the 'reference base/sequence' from `AICDA_mut.csv`. This will involve careful parsing of position information (single base vs. ranges).

## Implement mutation checking procedure

### Subtask:
Develop a function to iterate through each FASTA sequence and cross-reference it with the `df_mutations` data. For each mutation, the function will verify if the base or sequence at the given position(s) in the FASTA file matches the 'reference base/sequence' from `AICDA_mut.csv`. This will involve careful parsing of position information (single base vs. ranges).


**Reasoning**:
I need to import the `SeqIO` module from the `Bio` library to handle FASTA file parsing, as specified in the instructions.



In [ ]:
from Bio import SeqIO

print("SeqIO imported successfully.")

# Task
Update the `fasta_files` list with the correct filenames: `AB040431.1.fasta`, `AF529819.1.fasta`, `AY536516.1.fasta`, `BC006296.2.fasta`, and `BT007402.1.fasta`.

## Update FASTA file list

### Subtask:
The `fasta_files` list will be updated with the correct filenames: `AB040431.1.fasta`, `AF529819.1.fasta`, `AY536516.1.fasta`, `BC006296.2.fasta`, `BT007402.1.fasta`. This ensures the mutation checking procedure uses the correct input files. The `df_mutations` DataFrame, loaded from `AICDA_mut.csv`, is already available.


**Reasoning**:
The subtask requires updating the `fasta_files` list with the new, specified filenames. I will use a code block to reassign the `fasta_files` variable to the new list and print its content for verification.



In [ ]:
fasta_files = [
    'AB040431.1.fasta',
    'AF529819.1.fasta',
    'AY536516.1.fasta',
    'BC006296.2.fasta',
    'BT007402.1.fasta'
]

print(f"Updated fasta_files list: {fasta_files}")

In [ ]:
from Bio import SeqIO

# CDS offset: for AB040431.1 and BC006296.2, CDS starts at position 77 (1-indexed),
# so we add 76 to each mutation position to map it onto the FASTA coordinate.
CDS_OFFSET = {
    'AB040431.1.fasta': 76,
    'BC006296.2.fasta': 76,
}

def check_mutations_in_fasta(fasta_file, df_mutations):
    """
    Check each mutation in df_mutations against the FASTA sequence.

    For AB040431.1 and BC006296.2 the CDS begins at nucleotide 77,
    so the FASTA position = mutation_position + 76.
    For all other files no offset is applied.
    """
    try:
        record = SeqIO.read(fasta_file, "fasta")
        seq = str(record.seq).upper()
    except FileNotFoundError:
        print(f"File not found: {fasta_file}")
        return []

    offset = CDS_OFFSET.get(fasta_file, 0)
    results = []

    for _, row in df_mutations.iterrows():
        pos_str = str(row['position'])
        ref     = str(row['reference base/sequence']).upper()
        mut_type = row['mutation type']

        try:
            if '-' in pos_str:
                start, end = (int(x) for x in pos_str.split('-'))
                fasta_base = seq[start + offset - 1 : end + offset]   # 0-indexed slice
            else:
                pos        = int(pos_str)
                fasta_base = seq[pos + offset - 1]                     # 0-indexed
        except IndexError:
            fasta_base = "OUT_OF_BOUNDS"

        results.append({
            'fasta_file':        fasta_file,
            'mutation_type':     mut_type,
            'position':          pos_str,
            'reference_base_df': ref,
            'fasta_base_at_pos': fasta_base,
            'is_match':          fasta_base == ref,
        })

    return results


In [ ]:
fasta_files = [
    'AB040431.1.fasta',
    'AF529819.1.fasta',
    'AY536516.1.fasta',
    'BC006296.2.fasta',
    'BT007402.1.fasta',
]

all_results = []
for fasta_file in fasta_files:
    print(f"Checking {fasta_file}...")
    all_results.extend(check_mutations_in_fasta(fasta_file, df_mutations))

df_all_results = pd.DataFrame(all_results)

# A file is the correct AICDA reference if ALL positions match
summary = df_all_results.groupby('fasta_file')['is_match'].all().rename('is_right_AICDA_version')
df_all_results = df_all_results.join(summary, on='fasta_file')

display(df_all_results.head(10))
print("\nPer-file summary:")
display(summary.reset_index())

df_all_results.to_csv('all_AICDA_results.csv', index=False)
print("Results saved to all_AICDA_results.csv")


## Implement mutation checking procedure

### Subtask:
Develop a function to iterate through each FASTA sequence and cross-reference it with the `df_mutations` data. For each mutation, the function will verify if the base or sequence at the given position(s) in the FASTA file matches the 'reference base/sequence' from `AICDA_mut.csv`. This will involve careful parsing of position information (single base vs. ranges).


**Reasoning**:
I need to define the `check_mutations_in_fasta` function as described in the instructions. This function will read a FASTA file, iterate through the `df_mutations` DataFrame, parse mutation positions (single or range), extract the corresponding sequence from the FASTA file, and compare it against the reference sequence from the DataFrame, storing the results in a list of dictionaries.



**Reasoning**:
Now that the `check_mutations_in_fasta` function is defined, I will iterate through each FASTA file in the `fasta_files` list and apply this function with the `df_mutations` DataFrame. The results from each file will be collected and then combined into a single DataFrame for comprehensive analysis.



In [ ]:
df_all_results.to_csv('all_AICDA_results.csv', index=False)
print("df_all_results exported to 'all_AICDA_results.csv' successfully.")

## Process and report results for each FASTA file

### Subtask:
Analyze the `df_all_results` DataFrame to determine for each FASTA file whether all mutations match their reference sequences, indicating if the FASTA sequence is 'wild-type' at each mutation site. Display a summary of these findings.


## Summary:

### Q&A
1.  **Has the `fasta_files` list been updated with the correct filenames?**
    Yes, the `fasta_files` list has been successfully updated with the specified filenames: `AB040431.1.fasta`, `AF529819.1.fasta`, `AY536516.1.fasta`, `BC006296.2.fasta`, and `BT007402.1.fasta`.
2.  **Is there a mechanism to determine for each FASTA file whether all mutations match their reference sequences?**
    Yes, a function named `check_mutations_in_fasta` has been developed and applied to process all FASTA files against the mutation data. The results, indicating whether each specific mutation matches its reference in each FASTA file, are stored in `df_all_results`.

### Data Analysis Key Findings
*   The `fasta_files` list was successfully updated to include the five specified FASTA filenames for analysis.
*   A custom function, `check_mutations_in_fasta`, was developed to systematically compare mutation data from `df_mutations` with sequences in each FASTA file.
*   This function accurately parses both single-base positions (e.g., '334') and sequence ranges (e.g., '235-243') from the mutation data, converting them to 0-indexed positions for precise sequence extraction.
*   It performs a case-insensitive comparison between the extracted FASTA sequence and the 'reference base/sequence' from `df_mutations` to determine if a match exists.
*   The function includes robust error handling for common issues such as `FileNotFoundError`, invalid position strings, and out-of-bounds sequence access.
*   The `check_mutations_in_fasta` function was executed for each of the five FASTA files, and all results were aggregated into a comprehensive DataFrame named `df_all_results`.
*   The `df_all_results` DataFrame contains key columns such as `fasta_file`, `mutation_type`, `original_position_str`, `reference_base_df`, `fasta_sequence_at_pos`, and a boolean `is_match` indicating the outcome of each comparison.

### Insights or Next Steps
*   The current `df_all_results` DataFrame is ready to be used to determine the "wild-type" status for each FASTA file. The next step should involve grouping the results by `fasta_file` and checking if *all* `is_match` values for a given file are `True`.
*   Consider implementing a summary report that clearly states for each FASTA file whether it is considered "wild-type" (all mutations match reference) or if it contains any discrepancies.


# Batch Process All xxxbase Folders

## Task
Walk through all xxxbase folders, scrape the UniProt link from each `*pub.html`, and write results to `uniprot_link.csv`.

In [23]:
# helper for scraping UniProt IDs from HTML
import re

def extract_uniprot_from_html(file_path):
    """Return the UniProt accession found in a pub.html file.

    Only matches URLs pointing at the UniProt domain so OMIM or others
    are ignored. Tries utf-8 then latin-1 for encoding.
    """
    for enc in ('utf-8', 'latin-1'):
        try:
            with open(file_path, 'r', encoding=enc, errors='ignore') as f:
                content = f.read()
        except Exception:
            continue
        # look specifically for a uniprot.org entry URL
        m = re.search(r'https?://[^\s"\']*uniprot\.org/entry/([A-Z0-9]+)', content)
        if m:
            return m.group(1)
        # fallback: any UniProt: label
        m2 = re.search(r'UniProt:\s*([A-Z0-9]+)', content)
        if m2:
            return m2.group(1)
    return None


In [24]:
import os
import re
import pandas as pd
from pathlib import Path

# define base directory
BASE_DIR = Path("f:/idbase")

# locate xxxbase folders
base_folders = sorted([d for d in BASE_DIR.iterdir() if d.is_dir() and d.name.endswith('base')])

print(f"Found {len(base_folders)} xxxbase folders")
print("Scraping UniProt IDs from each pub.html...\n")

uniprot_results = []

for folder_path in base_folders:
    gene_name = folder_path.name.replace('base', '')
    pub_file = folder_path / f"{gene_name.lower()}pub.html"

    if not pub_file.exists():
        print(f"⚠ {gene_name}: {pub_file.name} not found")
        continue

    uniprot_id = extract_uniprot_from_html(pub_file)
    if uniprot_id:
        # store only the identifier portion, not full URL
        uniprot_results.append([gene_name, uniprot_id])
        print(f"✓ {gene_name}: {uniprot_id}")
    else:
        print(f"✗ {gene_name}: UniProt ID not found")

# save final CSV
df_uniprot = pd.DataFrame(uniprot_results, columns=["Gene/IDBase", "UniProt ID"])
df_uniprot.to_csv('uniprot_link.csv', index=False)
print("\nSaved all IDs to uniprot_link.csv")


Found 137 xxxbase folders
Scraping UniProt IDs from each pub.html...

✓ ADA: P00813
✓ AICDA: Q9GZX7
✓ AIRE: O43918
✓ AK2: Q8TCY3
✓ AP3B1: O00203
✓ BIRC4: P98170
✓ BLM: P54132
✓ BLNK: O75498
✓ BTK: Q06187
✗ C1QA: UniProt ID not found
✓ C1QB: P02746
✓ C1QC: P02747
✗ C1S: UniProt ID not found
✗ C2: UniProt ID not found
✗ C3: UniProt ID not found
✗ C5: UniProt ID not found
✗ C6: UniProt ID not found
✗ C7: UniProt ID not found
✓ C8B: P07358
✗ C9: UniProt ID not found
✓ CARD9: Q9H257
✓ CASP10: Q92851
✓ CASP8: Q14790
✓ CD19: P15391
✓ CD247: P20963
✓ CD3D: P04234
✓ CD3E: P07766
✓ CD3G: P09693
✓ CD40: P25942
✓ CD40L: P29965
✓ CD55: P08174
✓ CD59: P13987
✓ CD79A: P11912
✓ CD79B: P40259
✓ CD8A: P01732
✓ CEBPE: Q15744
✓ CFD: P00746
✓ CFH: P08603
✓ CFI: P05156
✓ CFP: P27918
✓ CIITA: P33076
✓ CTSC: P53634
✓ CXCR4: P30991
✓ CYBA: P13498
✗ CYBB: UniProt ID not found
✓ DCLRE1C: Q96SD1
✓ DKC1: O60832
✓ DNMT3B: Q9UBC3
✓ ELA2: P08246
✓ FASLG: P48023
✓ FCGR1A: P12314
✓ FCGR3A: P08637
✓ FERMT3: Q86UX7
✓ FOX